# 03. MCTS engine

This component is the AlphaZero-style search itself. It is built from small,
swappable seams, none of which know anything about Allen intervals:

* `expansion` (`RuleExpansion`) asks the grammar for the legal productions and
  adds one child at a time.
* `selection` (`PUCTSelection`) picks which child to descend into, using each
  child's visit count, value, and `prior`.
* `backprop` (`MaxRewardBackup`, `PercentileRewardBackup`) carries a leaf value
  back up the tree and runs the dead-branch cascade.
* `value_target` (`MaxValue`, `ExpectedValue`, ...) reads each step's value target
  off the tree, paired to the backup operator.
* `replay` (`Trajectory`, `ReplayBuffer`) records one self-play episode and turns
  it into training rows with scaled value targets.
* `self_play` (`run_self_play`) ties them together into one episode.

Because the engine only talks to the grammar through the protocol from component
02, the same `run_self_play` drives a non-Allen grammar unchanged (last
section).

In [1]:
import alpha_rule
print("alpha_rule", alpha_rule.__version__)

alpha_rule 0.1.0


## Elements

`alpha_rule.mcts` re-exports the whole engine:

* `MCTSRuleNode`: the pure-state node from component 02.
* `ExpansionStrategy` / `RuleExpansion`: add one child via the grammar.
* `SelectionStrategy` / `PUCTSelection`: PUCT with KataGo-style first-play
  urgency; reads `child.prior`, `child.Q_max` (or the filtered mean), and visit
  counts.
* `BackpropStrategy` / `MaxRewardBackup` / `PercentileRewardBackup`: value backup
  strategies; both run the same dead-branch cascade.
* `ValueTarget` / `MaxValue` / `ExpectedValue` / `MeanPercentileValue` /
  `RealizedReturn`: how each step's value target `z_t` is read off the tree.
  `default_value_target(backup)` picks the one that matches the backup.
* `Trajectory` / `TrajectoryStep` / `ReplayBuffer`: one episode and its training
  rows; `Trajectory.value_targets()` scales them into `[-1, +1]`.
* `run_self_play(grammar, simulator, ...)`: one self-play episode, returning a
  `Trajectory`.

The leaf value comes from a `simulator` you pass in (any object with
`evaluate(node)`), so the engine has no environment or network baked in.

## The seams

### Expansion

`RuleExpansion` adds one child per call, in the grammar's production order, and
returns `None` once every production has a child.

In [2]:
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.mcts import RuleExpansion

g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<", ">"))
root = g.root()
exp = RuleExpansion(g)

children = []
while True:
    child = exp.expand(root)
    if child is None:
        break
    children.append(child.name)
print("root expands into:", children)
print("root fully expanded:", root.is_fully_expanded())
print("further expand() returns None:", exp.expand(root) is None)

root expands into: ['A', 'B']
root fully expanded: True
further expand() returns None: True


### Backpropagation

`MaxRewardBackup` walks from the simulated leaf to the root, incrementing each
node's visit count `N` and raising `Q_max` toward the value. A `-inf` value marks
only that leaf dead. A parent dies only once it is fully expanded and all its
children are dead (the dead-branch cascade), so one bad rollout never kills a
whole path.

In [3]:
from alpha_rule.mcts import MaxRewardBackup

g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<",))
root = g.root()
exp = RuleExpansion(g)
a = exp.expand(root)            # root -> "A"
leaf = exp.expand(a)            # "A" -> "A <END>"  (END_RULE is first for non-root)

backup = MaxRewardBackup()
backup.update(leaf, 2.0)
backup.update(leaf, 5.0)        # Q_max rises to 5 and never falls
print(f"leaf : N={leaf.N}  Q_max={leaf.Q_max:.1f}")
print(f"A    : N={a.N}  Q_max={a.Q_max:.1f}")
print(f"root : N={root.N}  Q_max={root.Q_max:.1f}")

backup.update(leaf, float("-inf"))
print("after -inf, leaf is_dead:", leaf.is_dead,
      "| A is_dead:", a.is_dead, "(A is not fully expanded yet)")

leaf : N=2  Q_max=5.0
A    : N=2  Q_max=5.0
root : N=2  Q_max=5.0
after -inf, leaf is_dead: True | A is_dead: False (A is not fully expanded yet)


### Selection

`PUCTSelection` scores each child with `Q + c_puct * prior * sqrt(sum_N) /
(1 + N)`. On the first visit every child is unvisited, so the score is driven by
its `prior` (which a network would set). Dead children score `-inf` and are never
picked.

In [4]:
from alpha_rule.mcts import PUCTSelection

g = AllenIntervalGrammar(event_types=("A", "B", "C"), relations=("<",))
root = g.root()
exp = RuleExpansion(g)
while not root.is_fully_expanded():
    exp.expand(root)                        # children A, B, C

for child, prior in zip(root.children, (0.1, 0.7, 0.2)):
    child.prior = prior                     # as a NeuralEvaluator would set them

sel = PUCTSelection(c_puct=1.5)
for c in root.children:
    print(f"  {c.name}: prior={c.prior:.2f}  score={sel.score(root, c):+.3f}")
print("selected (highest score):", sel.select(root).name)

  A: prior=0.10  score=+0.150
  B: prior=0.70  score=+1.050
  C: prior=0.20  score=+0.300
selected (highest score): B


## One self-play episode

`run_self_play` runs `n_simulations` simulations per construction step, samples
the next production from the visit counts, applies it, and records a
`TrajectoryStep`. It stops at `depth_limit` or a terminal. The leaf value comes
from the `simulator` you pass. Here is a tiny deterministic one, with no
environment: it rewards a rule for each `A` and `<` token, so the search is
nudged toward them. `reward_scale` is its positive reward cap, read later to
scale the value targets.

In [5]:
import numpy as np
from alpha_rule.mcts import run_self_play

class PreferenceSimulator:
    # Deterministic, environment-free reward: count of 'A' and '<' tokens.
    reward_scale = 4.0          # positive reward cap, read by run_self_play

    def __init__(self):
        self.calls = 0

    def evaluate(self, node):
        self.calls += 1
        toks = node.name.replace("<END>", "").split()
        return float(sum(t in ("B", "<") for t in toks))

g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<", ">"))
sim = PreferenceSimulator()
traj = run_self_play(
    grammar=g,
    simulator=sim,
    n_simulations=40,
    depth_limit=4,
    rng=np.random.default_rng(0),
)
print("simulator calls:", sim.calls, "| trajectory steps:", len(traj.steps))
for i, s in enumerate(traj.steps):
    print(f"  step {i}: {s.state!r:16} -> {s.next_state!r:14} reward={s.reward:+.1f}")
print("visit_pi at step 0 sums to:", round(sum(traj.steps[0].visit_pi.values()), 6))

simulator calls: 164 | trajectory steps: 4
  step 0: '<ROOT>'         -> 'B'            reward=+1.0
  step 1: 'B'              -> 'B B'          reward=+2.0
  step 2: 'B B'            -> 'B B <'        reward=+3.0
  step 3: 'B B <'          -> 'B B < B'      reward=+4.0
visit_pi at step 0 sums to: 1.0


## Value-target strategies

The value head should regress the same quantity the search optimises, so the value
target is its own seam. `default_value_target` pairs it to the backup:
`MaxRewardBackup` to `MaxValue` (`node.Q_max`, "best rule reachable"), and
`PercentileRewardBackup` to `ExpectedValue` (visit-weighted mean over live
children of their filtered mean). `MeanPercentileValue` (the node's own filtered
mean) and `RealizedReturn` (the simulator's eval stamped on each chosen step) are
also available. Every tree-derived target skips dead children, so the value target
and the policy `visit_pi` agree on which branches exist.

In [6]:
from alpha_rule.mcts import (
    MaxValue, ExpectedValue, MeanPercentileValue, RealizedReturn,
    default_value_target, MaxRewardBackup, PercentileRewardBackup, MCTSRuleNode,
)

print("default for MaxRewardBackup       :", type(default_value_target(MaxRewardBackup())).__name__)
print("default for PercentileRewardBackup:", type(default_value_target(PercentileRewardBackup())).__name__)

# A parent with two live children, to read the four targets off.
parent = MCTSRuleNode(name="P", n_possible_actions=2)
for nm, n, qmax, qsum, npass in [("A", 3, 0.8, 1.5, 3), ("B", 1, 0.4, 0.4, 1)]:
    c = MCTSRuleNode(name=nm, parent=parent, parent_action=nm)
    c.N, c.Q_max, c.Q_sum, c.N_passers = n, qmax, qsum, npass
    parent.children.append(c)
parent.Q_max, parent.Q_sum, parent.N_passers = 0.8, 1.6, 4
parent.realized_reward = 0.6

print("MaxValue            (node.Q_max)          :", MaxValue().state_value(parent))
print("ExpectedValue       (mean over children)  :", round(ExpectedValue().state_value(parent), 4))
print("MeanPercentileValue (node Q_sum/N_passers):", round(MeanPercentileValue().state_value(parent), 4))
print("RealizedReturn      (node.realized_reward):", RealizedReturn().state_value(parent))

default for MaxRewardBackup       : MaxValue
default for PercentileRewardBackup: ExpectedValue
MaxValue            (node.Q_max)          : 0.8
ExpectedValue       (mean over children)  : 0.475
MeanPercentileValue (node Q_sum/N_passers): 0.4
RealizedReturn      (node.realized_reward): 0.6


## Value targets and the replay buffer

Each step's value target `z_t` is the `state_value` the strategy read off the tree,
scaled into the tanh value head's `[-1, +1]` range by a single clip against the
simulator's positive reward cap `value_scale`:

    z = clip(state_value / value_scale, -1, +1)

Good rules spread across `[0, 1]`; the negative tail and `-inf` structural failures
collapse onto `-1`. `value_scale` is read from `simulator.reward_scale` (here
`4.0`) when you do not pass it, and stamped on the `Trajectory`. A `ReplayBuffer`
stores one row per step and reports `fill_fraction`, so you can size its `capacity`
to your self-play throughput.

In [7]:
from alpha_rule.mcts import Trajectory, TrajectoryStep, ReplayBuffer

print("self-play traj value_scale (from simulator.reward_scale):", traj.value_scale)

# A hand-built trajectory makes the clip and the fallbacks easy to read. With a
# reward cap of 2.0, z = clip(state_value / 2, -1, +1).
demo = Trajectory(
    steps=[
        TrajectoryStep(state="s0", visit_pi={"a": 1.0}, reward=1.5, next_state="s1", state_value=1.8),
        TrajectoryStep(state="s1", visit_pi={"a": 1.0}, reward=0.4, next_state="s2", state_value=0.4),
        TrajectoryStep(state="s2", visit_pi={"a": 1.0}, reward=-1.0, next_state="s3", state_value=-3.0),
        TrajectoryStep(state="s3", visit_pi={"a": 1.0}, reward=0.2, next_state="s4", state_value=None),
    ],
    value_scale=2.0,
)
print("raw state_value :", [s.state_value for s in demo.steps])
print("scaled target z :", [round(z, 3) for z in demo.value_targets()])
print("  -3.0 clips to -1 (an -inf would land there too); the None step falls back")
print("  to its own reward, 0.2 / 2 = 0.1.")

buf = ReplayBuffer(capacity=100)
buf.push_trajectory(demo)
print("buffer rows:", len(buf), "| capacity:", buf.capacity, "| fill_fraction:", round(buf.fill_fraction, 3))

self-play traj value_scale (from simulator.reward_scale): 4.0
raw state_value : [1.8, 0.4, -3.0, None]
scaled target z : [0.9, 0.2, -1.0, 0.1]
  -3.0 clips to -1 (an -inf would land there too); the None step falls back
  to its own reward, 0.2 / 2 = 0.1.
buffer rows: 4 | capacity: 100 | fill_fraction: 0.04


## Backup strategies: max vs percentile

`MaxRewardBackup` counts every finite sample in its filtered mean.
`PercentileRewardBackup` keeps only samples at or above a percentile of the node's
own history, so a single low outlier does not drag the value down (and,
symmetrically, a single lucky spike does not peg it on noisy simulators).

In [8]:
from alpha_rule.mcts import MaxRewardBackup, PercentileRewardBackup, MCTSRuleNode

def one_leaf():
    parent = MCTSRuleNode(name="P", n_possible_actions=2)   # not fully expanded
    leaf = MCTSRuleNode(name="P A", parent=parent, parent_action="A")
    parent.children.append(leaf)
    return leaf

rewards = [10.0, 10.0, 10.0, 10.0, 1.0]      # four good, one low outlier

lmax = one_leaf()
mb = MaxRewardBackup()
for r in rewards:
    mb.update(lmax, r)

lpct = one_leaf()
pb = PercentileRewardBackup(percentile=50, min_samples=3)
for r in rewards:
    pb.update(lpct, r)

print(f"max        : N={lmax.N}  passers={lmax.N_passers}  "
      f"filtered_mean={lmax.Q_sum / lmax.N_passers:.2f}")
print(f"percentile : N={lpct.N}  passers={lpct.N_passers}  "
      f"filtered_mean={lpct.Q_sum / lpct.N_passers:.2f}  (low outlier dropped)")

max        : N=5  passers=5  filtered_mean=8.20
percentile : N=5  passers=4  filtered_mean=10.00  (low outlier dropped)


## Swapping the grammar drives the same search

The engine only uses the grammar protocol from component 02, so the same
`run_self_play` works on a grammar with no Allen logic. Here is the binary-string
`ToyGrammar` again, driven by a length-reward simulator. Only the toy's own tokens
appear in the trajectory. This is the seam that makes the search reusable; it is
also pinned by `test_custom_grammar_drives_self_play_unchanged`.

In [9]:
from alpha_rule.grammar.production import Production
from alpha_rule.mcts import MCTSRuleNode, run_self_play

class ToyGrammar:
    TOKENS = ("0", "1")
    MAX_LEN = 3

    def _new(self, name, level, parent=None, parent_action=None, is_terminal=False):
        node = MCTSRuleNode(name=name, level=level, parent=parent,
                            parent_action=parent_action, is_terminal=is_terminal)
        node.n_possible_actions = len(self.applicable_productions(node))
        return node

    def root(self):
        return self._new("<ROOT>", 0)

    def vocab(self):
        return list(self.TOKENS) + ["END"]

    def applicable_productions(self, state):
        if state.is_terminal or state.level >= self.MAX_LEN:
            return []
        prods = [Production(name=t, kind="token") for t in self.TOKENS]
        if state.name != "<ROOT>":
            prods = [Production(name="END", kind="stop")] + prods
        return prods

    def apply(self, state, production):
        is_term = production.kind == "stop"
        base = "" if state.name == "<ROOT>" else state.name
        name = (base + " <END>").strip() if is_term else (base + " " + production.name).strip()
        child = self._new(name, state.level + 1, parent=state,
                          parent_action=production.name, is_terminal=is_term)
        state.children.append(child)
        return child

    def is_terminal(self, state):
        return state.is_terminal

class LengthSimulator:
    # Reward = number of tokens in the rule (favours longer strings).
    def evaluate(self, node):
        return float(len([t for t in node.name.replace("<END>", "").split() if t]))

traj = run_self_play(
    grammar=ToyGrammar(),
    simulator=LengthSimulator(),
    n_simulations=16,
    depth_limit=3,
    rng=np.random.default_rng(0),
)
allowed = set(ToyGrammar().vocab()) | {"<END>"}
used = sorted({t for s in traj.steps if s.next_state for t in s.next_state.split()})
print("steps       :", len(traj.steps))
print("tokens used :", used)
print("only toy tokens, no Allen symbols leaked in:", set(used) <= allowed)

steps       : 3
tokens used : ['0', '<END>']
only toy tokens, no Allen symbols leaked in: True


## Basic checks

Asserts that mirror the engine's unit tests. Run the cell; `ok` means the seams
behave as expected.

In [10]:
import numpy as np
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.mcts import (
    RuleExpansion, MaxRewardBackup, PercentileRewardBackup, PUCTSelection,
    run_self_play, ReplayBuffer, MCTSRuleNode, MaxValue, ExpectedValue,
    default_value_target,
)

# Expansion: one child per production, in order, then None.
g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<",))
root = g.root()
exp = RuleExpansion(g)
names = []
while (child := exp.expand(root)) is not None:
    names.append(child.name)
assert names == ["A", "B"] and exp.expand(root) is None

# Backprop: Q_max never decreases; every ancestor's N counts.
parent = MCTSRuleNode(name="P", n_possible_actions=2)
leaf = MCTSRuleNode(name="P A", parent=parent, parent_action="A")
parent.children.append(leaf)
mb = MaxRewardBackup()
mb.update(leaf, 3.0)
mb.update(leaf, 1.0)
assert leaf.Q_max == 3.0 and leaf.N == 2 and parent.N == 2

# -inf marks only the leaf dead (parent is not fully expanded, so it survives).
mb.update(leaf, float("-inf"))
assert leaf.is_dead and not parent.is_dead

# Dead children score -inf and are never selected.
sel = PUCTSelection()
assert sel.score(parent, leaf) == float("-inf")

# Value-target defaults match the backup operator.
assert isinstance(default_value_target(MaxRewardBackup()), MaxValue)
assert isinstance(default_value_target(PercentileRewardBackup()), ExpectedValue)

# Self-play returns a bounded, valid trajectory; visit_pi sums to 1.
class _Sim:
    reward_scale = 1.0
    def evaluate(self, node):
        return 1.0
traj = run_self_play(grammar=g, simulator=_Sim(), n_simulations=8,
                     depth_limit=3, rng=np.random.default_rng(0))
assert 1 <= len(traj.steps) <= 3
assert all(abs(sum(s.visit_pi.values()) - 1.0) < 1e-9 for s in traj.steps)

# Replay buffer: one row per step; value targets clipped into [-1, 1].
buf = ReplayBuffer(capacity=50)
buf.push_trajectory(traj)
assert len(buf) == len(traj.steps)
zs = traj.value_targets()
assert all(np.isfinite(z) and -1.0 <= z <= 1.0 for z in zs)

print("ok")

ok


## Full unit tests

```
python -m alpha_rule.tests.run_tests test_expansion
python -m alpha_rule.tests.run_tests test_max_backup
python -m alpha_rule.tests.run_tests test_percentile_backup
python -m alpha_rule.tests.run_tests test_puct_selection
python -m alpha_rule.tests.run_tests test_filtered_mean
python -m alpha_rule.tests.run_tests test_value_target
python -m alpha_rule.tests.run_tests test_replay_buffer
python -m alpha_rule.tests.run_tests test_self_play
```

A few `self_play` / `percentile_backup` cases that call the training orchestrator
skip until component 06 lands `train()`.